# J4 Après-Midi | Générer des explications avec un LLM

**Instructrice :** Flore Vancompernolle Vromman

**Durée estimée :** 30 min

**Objectifs d'apprentissage :**
1. Construire un prompt à partir de données issues d'un modèle de machine learning.
2. Appeler l'API OpenAI pour générer du texte.
3. Personnaliser les explications produites.

**Prérequis :**
- Avoir suivi la présentation sur l'explicabilité (session précédente).
- Avoir une clé API OpenAI (fournie par les instructeurs).

## 1. Pourquoi utiliser un LLM pour expliquer une prédiction ?

En machine learning, on distingue deux grandes familles de modèles en ce qui concerne l'explicabilité :

- Les **modèles interprétables** (comme la régression linéaire ou ridge) : leurs paramètres sont directement lisibles. Les coefficients indiquent l'importance et la direction de l'effet de chaque variable sur la prédiction.
- Les **modèles black box** (comme les random forest ou réseaux de neurones) : leur fonctionnement interne est opaque. Pour les expliquer, on applique une couche supplémentaire comme **SHAP** ou **LIME**, qui reconstituent l'influence de chaque variable après coup.

Dans les deux cas, ces méthodes produisent des **sorties techniques** : coefficients, valeurs SHAP, graphiques, classements de variables. Ces informations sont précieuses pour le chercheur ou le développeur, mais difficiles à communiquer directement à un utilisateur final.

C'est là qu'intervient le LLM : il peut transformer ces données chiffrées en un texte personnalisé et compréhensible.

```
Modèle ML → coefficients × réponses → LLM → Explication en langage naturel
```

## 2. Setup

Pour appeler le LLM, nous utilisons l'**API OpenAI**, comme dans les notebooks précédents.

> ⚠️ **Sécurité :** Ne partagez jamais votre clé API publiquement. Dans un vrai projet, stockez-la dans un fichier `.env` (jamais dans le code).

In [ ]:
from openai import OpenAI

# Collez votre clé API ici (ne la partagez pas !)
api_key = "VOTRE_CLE_API_ICI"

client = OpenAI(api_key=api_key)

print("Client OpenAI initialisé !")

## 3. Les données de l'utilisateur

Nous reprenons l'exemple de **Datagotchi Santé** : un modèle de régression ridge prédit un score de bien-être entre 0 et 100 à partir de 20 questions sur les habitudes de vie.

Pour expliquer le score, on se concentre sur les **5 variables avec les coefficients positifs les plus élevés**, c'est-à-dire celles qui ont le plus d'impact favorable sur le score.

Les réponses de l'utilisateur sont **normalisées entre 0 et 1** (0 = réponse minimale, 1 = réponse maximale).

Le modèle a prédit un score de **56/100** pour cet utilisateur.

In [ ]:
# Les 5 variables les plus importantes (coefficient positif le plus élevé)
coefficients = {
    "sleep_quality": 19.48,
    "healthy_diet": 13.74,
    "activities_friends": 10.57,
    "neighborhood_friendly": 10.37,
    "volunteering": 7.58,
}

# Les réponses de l'utilisateur, normalisées entre 0 et 1
answers = {
    "sleep_quality": 0.7,
    "healthy_diet": 0.67,
    "activities_friends": 0.75,
    "neighborhood_friendly": 0.33,
    "volunteering": 0.0,
}

# Score de bien-être prédit par le modèle (sur 100)
score_total = 56

print(f"Score prédit : {score_total}/100")
print("\nVariables et réponses :")
for variable, reponse in answers.items():
    print(f"  {variable} : {reponse}")

## 4. Construire le prompt

Nous disposons maintenant de données chiffrées sur notre utilisateur. L'objectif est de passer de ces chiffres à une explication lisible et personnalisée.

Pour cela, nous allons construire un **prompt** qui intègre ces données. Un bon prompt doit contenir :
1. **Un rôle** : qui est le LLM dans ce contexte ?
2. **Le contexte** : les données pertinentes (score, variables, réponses)
3. **La tâche** : ce qu'on attend du LLM
4. **Le format** : longueur, ton, structure de la réponse

Nous calculons d'abord la **contribution** de chaque variable, c'est-à-dire son impact réel sur le score pour cet utilisateur :

```
contribution = coefficient × réponse normalisée
```

Par exemple, pour `sleep_quality` : 19.48 × 0.7 = **13.64 points**.

In [ ]:
# Calcul des contributions et construction du contexte pour le prompt
lignes = []
for variable, coeff in coefficients.items():
    reponse = answers[variable]
    contribution = round(coeff * reponse, 2)
    lignes.append(f"- {variable} : coefficient = {coeff}, réponse = {reponse}/1, contribution au score = {contribution} pts")

contexte = "\n".join(lignes)

# Construction du prompt complet
prompt = f"""Tu es un assistant de l'application Datagotchi Santé.
Ton rôle est d'expliquer à un utilisateur son score de bien-être de façon personnalisée et bienveillante.

L'utilisateur a obtenu un score de bien-être de {score_total}/100.

Voici les 5 facteurs les plus importants du modèle et les réponses de l'utilisateur :
{contexte}

Génère une explication personnalisée de ce score en 3-4 phrases.
Mentionne ses points forts et le point à améliorer le plus impactant.
Utilise un ton encourageant."""

# On affiche le prompt pour vérifier ce qu'on envoie au LLM
print(prompt)

## 5. Appeler le LLM

L'API OpenAI fonctionne comme dans les notebooks précédents :
- On envoie une liste de **messages** avec un rôle (`user`) et du contenu
- On choisit un **modèle** (ici `gpt-4o-mini`, rapide et économique)
- On fixe un nombre maximum de **tokens** en sortie

In [ ]:
def generer_explication(prompt, model="gpt-4o-mini", max_tokens=400):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

# Appel au LLM et affichage du résultat
explication = generer_explication(prompt)
print(explication)

### Hack-Time 🛠️

Choisissez une (ou plusieurs) des modifications suivantes :

**Option A — Changer le ton ou le format :**
- Rendez l'explication plus technique (mentionnez les chiffres de contribution)
- Générez l'explication en anglais
- Demandez une explication en une seule phrase courte

**Option B — Ajouter une recommandation :**
- Demandez au LLM d'identifier le **levier le plus efficace** pour améliorer le score
- Ajoutez une suggestion concrète pour la variable avec la contribution la plus faible

Modifiez le prompt dans la cellule ci-dessous et testez votre version !

In [ ]:
# Modifiez ce prompt selon l'option choisie !
nouveau_prompt = f"""Tu es un assistant de l'application Datagotchi Santé.
L'utilisateur a obtenu un score de {score_total}/100.

... # À vous de jouer !
"""

nouvelle_explication = generer_explication(nouveau_prompt)
print(nouvelle_explication)

## 6. Limites et enjeux

**La qualité dépend du prompt** : un prompt vague ou mal structuré produit des explications génériques ou incohérentes. C'est le principe *garbage in, garbage out*.

**Risque de confabulation** : le LLM peut inventer des liens de causalité ("votre qualité de sommeil *cause* votre anxiété") qui ne sont pas dans les données. Le modèle prédit des corrélations statistiques, pas des causalités.

**Enjeu de transparence et de responsabilité** : si un utilisateur prend des décisions de santé sur la base d'une explication générée par un LLM, qui est responsable en cas d'erreur ? C'est l'une des questions soulevées par l'**AI Act** sur les systèmes IA à impact sur les individus.